# Tutorial -  Volatilidad
Sergio Cabrales, Universidad de los Andes

https://www.sac2.com/

## 1. Carga de librerías, funciones y APIs necesarias.

#### 1.1. Instalan las librerías que no incluye Google Colab

In [ ]:
pip install yfinance

In [ ]:
pip install mplfinance

In [ ]:
pip install arch

#### 1.2. Se cargan las librerías requeridas

In [ ]:
# Funciones numéricas adicionales
import numpy as np

# Lectura de datos y manejo de Data-sets
import pandas as pd

# Datos
import yfinance as yfin

# Gráficos
import matplotlib.pyplot as plt
from statsmodels.graphics.tsaplots import plot_acf, plot_pacf
import statsmodels.api as sm
import seaborn as sns

#analisis tecnico
import mplfinance as mpf

# Probabilidad y estadística
import math
from scipy.stats import norm, chi2, jarque_bera,shapiro
from scipy.optimize import brentq
from scipy import stats

# ARCH model
from arch import arch_model

# Engel test for ARCH effects
from statsmodels.stats.diagnostic import het_arch

## 2. Obtención de datos históricos

#### 2.1. Descarga de datos desde Yahoo Finance

https://finance.yahoo.com/


In [ ]:
# Descargamos datos de la acción sleccionada:
df = yfin.download('AMZN', start='2010-01-01', multi_level_index=False)
df

## 3. Visualización y Estadísticas Descriptivas

### 3.1. Utiliza la librería mpf para hacer un gráfico de velas japonesas de la acción

In [ ]:
mpf.plot(df,type='candle', volume=True,figratio=(19,8),style='yahoo',title='Amazon')

## 4. Retornos

### 4.1. Retornos Logarítmicos

Los retornos logarítmicos se calculan como:
$$
r_{t} = ln \left( \frac{S_t}{S_{t-1}} \right ) = ln \left( S_{t} \right) - ln \left( S_{t-1} \right)
$$

In [ ]:
# Guardamos los retornos logaritmicos en una nueva columna.
df['Log Returns'] = np.log(df['Close']) - np.log(df['Close'].shift(1))
df['Log Returns'][0] = 0
df

### 4.3. Retornos Logarítmicos anualizados

Podemos calcular el log-retorno anual ($r$) como el número de días bursátiles (252 días) por el promedio del log-retorno diario:

$$
r = 252 \bar{r_t}
$$

In [ ]:
# Podemos imprimir el retornos anual:
LogReturns = np.mean(df["Log Returns"])*252
LogReturns

### 4.4. Gráfica de retornos
- Podemos graficar los retornos igual que como graficamos los precios.

In [ ]:
# Gráfico de los retornos logarítmicos
plt.figure(figsize=(15,8))
plt.plot(df['Log Returns'], color = 'blue')
plt.title('Retornos Logarítmicos')
plt.xlabel('Fecha')
plt.show()

## 5. Volatilidad

### 5.1 Volatilidad diaria y anual

La volatilidad diaria del activo es la desviación estándar de sus retornos o la raíz de la varianza:  

$$vol=desv(r)=\sqrt{Var(r)}$$

En finanzas, se utiliza con mayor frecuencia la volatilidad anualizada ($\sigma$) en lugar de la volatilidad diaria. Teniendo en cuenta que en cada año hay 252 días bursátiles:

$$ \sigma^{2} = \sum_{1}^{252} Var_{diaria}$$
$$ \sigma^{2} = 252 \sigma_{diaria}^{2}$$

Se saca la raíz cuadra a ambos lados para calcular la volatilidad:

$$ \sqrt{\sigma^{2}} = \sqrt{252 \sigma_{diaria}^{2}}$$
$$ \sigma = \sigma_{diaria} \sqrt{252}$$

In [ ]:
# Calculamos la volatilidad diaria con los retornos logaritmicos.
vol_d = np.std(df['Log Returns'])

# Anualizamos la volatilidad diaria.
vol_a = vol_d * np.sqrt(252)

print("Volatilidad diaria: {:.4f} %".format(100*vol_d))
print("Volatilidad anualizada: {:.4f} %".format(100*vol_a))

## 6. Normalidad de los retornos

### 6.1. Histogram

In [ ]:
# Create a histogram with k bins
k = int(math.sqrt(len(df['Log Returns'])))

plt.hist(df['Log Returns'], bins=k)

# Add labels and a title
plt.xlabel('Values')
plt.ylabel('Frequency')
plt.title('Histogram of Log-Returns')

# Show the plot
plt.show()

### 6.2. Q-Q Plot

In [ ]:
# Se crea una Q-Q plot con el paquete statsmodels.api
mean = np.mean(df['Log Returns'])
std_dev = np.std(df['Log Returns'])
n = len(df['Log Returns'])
sm.qqplot(df['Log Returns'], stats.norm, loc=mean, scale=std_dev, line='45')
# Se agrega un título a la gráfica
plt.title("Q-Q Plot")
# Se muestra la gráfica
plt.show()

### 6.3. P-P Plot

In [ ]:
# Se crea una PP plot calculando las probabilidades empíricas y teóricas
mean = np.mean(df['Log Returns'])
std_dev = np.std(df['Log Returns'])
n = len(df['Log Returns'])
fig, ax = plt.subplots()
# Se calculan las probabilidades empíricas
p = np.arange(1, n + 1) / n - 0.5 / n
# Se calculan las probabilidades teóricas
pp = np.sort(stats.norm.cdf(df['Log Returns'],loc=mean, scale=std_dev ))
sns.scatterplot(x=pp, y=p, color='blue', edgecolor='blue', ax=ax)
ax.set_title('P-P plot')
ax.set_xlabel('Theoretical Probabilities')
ax.set_ylabel('Sample Probabilities')
ax.margins(x=0, y=0)
# Se dibuja la línea roja de 45°
plt.plot(np.linspace(0, 1.01), np.linspace(0, 1.01), 'r', lw=2)
# Se muestra la gráfica
plt.show()

### 6.4. Jarque-Bera test

In [ ]:
# perform the Jarque-Bera test
jb_value, p_value = jarque_bera(df['Log Returns'])

# print the results
print("Jarque-Bera value: ", jb_value)
print("p-value: ", p_value)

if p_value > 0.05:
    print("The data is normally distributed")
else:
    print("The data is not normally distributed")

## 7. Autocorrelación

### 7.1. Autocorrelation

In [ ]:
# Crear el autocorrelograma
plot_acf(df['Log Returns'], lags = 21)
plt.show()

In [ ]:
df['Log Returns^2'] = df['Log Returns']**2

In [ ]:
# Crear el autocorrelograma de retornos al cuadrado
plot_acf(df['Log Returns^2'], lags = 21)
plt.show()

### 7.2. Partial Autocorrelation

In [ ]:
# Crear el autocorrelograma parcial
plot_pacf(df['Log Returns'], lags = 21)
plt.show()

In [ ]:
# Crear el autocorrelograma parcial
plot_pacf(df['Log Returns^2'], lags = 21)
plt.show()

### 7.3. Ljung-Box test

In [ ]:
def ljung_box(x, lags):
  #   Performs the Ljung-Box test for autocorrelation in a time series.
  #   :param x: the time series data
  #   :param lags: the number of lags to use in the test
  #   :param alpha: the significance level
  #   :return: the test statistic and p-value
  n = len(x)
  Q = n * (n + 2) * np.sum([(np.corrcoef(x[:-i], x[i:])[0, 1])**2 / (n - i) for i in range(1, lags + 1)])
  df = lags
  p_value = 1 - chi2.cdf(Q, df)
  return Q, p_value

In [ ]:
ljung_box(df['Log Returns'],2)

## 8. ARCH effects

### 8.2. LJung - Box Test

In [ ]:
ljung_box(df['Log Returns^2'],5)

## 9. GARCH model

### 9.1. GARCH model

$ \sigma_{t}^{2} = \alpha_{0} + \sum^{q}_{j=1} \alpha_j ϵ_{t-j}^{2} +  \sum^{p}_{i=1} \beta_i \sigma_{t-i}^{2} $

In [ ]:
# Create the ARCH model
model = arch_model(df['Log Returns'], mean='Zero', vol='GARCH', q=1, p=1, rescale=False)

# Fit the model
results = model.fit()

# Print the results summary
print(results.summary())

In [ ]:
params = pd.DataFrame(results.params)
params

In [ ]:
# get residuals
e = pd.DataFrame(results.std_resid)
e['resid^2'] = e['std_resid']**2
e

### 9.2. Histogram of Residuals

In [ ]:
# Create a histogram with k bins
k = int(math.sqrt(len(e['std_resid'])))

plt.hist(e['std_resid'], bins=k)

# Add labels and a title
plt.xlabel('Values')
plt.ylabel('Frequency')
plt.title('Histogram of Residuals')

# Show the plot
plt.show()

### 9.3. Q-Q plot - Residuals

In [ ]:
# Se crea una Q-Q plot con el paquete statsmodels.api
pplot = sm.ProbPlot(e['std_resid'], stats.norm, fit=True)
# Se agrega un título a la gráfica
fig = pplot.qqplot(line="45")
plt.title("Q-Q Plot")
# Se muestra la gráfica
plt.show()

### 9.4. P-P plot - Residuals

In [ ]:
# Se crea una P-PQ plot
pplot = sm.ProbPlot(e['std_resid'], stats.norm, fit=True)
# Se agrega un título a la gráfica
fig = pplot.ppplot(line="45")
plt.title("P-P Plot")
# Se muestra la gráfica
plt.show()

### 9.5. Autocorrelation - Residuals

In [ ]:
# Crear el autocorrelograma
plot_acf(e['std_resid'], lags = 21)
plt.show()

In [ ]:
# Crear el autocorrelograma
plot_acf(e['resid^2'], lags = 21)
plt.show()

### 9.6. Jarque Bera Test - Residuals

In [ ]:
# perform Jarque-Bera of residuals
stats.jarque_bera(e['std_resid'])

### 9.7. Box-Ljung test - Squared.Residuals

In [ ]:
# perform Box-Ljung test
p_value = ljung_box(e['resid^2'],5)[1]
# print the results
print("Box-Ljung test p-value: {:.4f}".format(p_value))

### 9.8. Volatility - Forecast

In [ ]:
fig = results.plot(annualize="D")

In [ ]:
df['Volatility'] = results.conditional_volatility*math.sqrt(252)
# Gráfico de la volatilidad
plt.figure(figsize=(15,8))
plt.plot(df['Volatility'], color = 'orange')
plt.title('Volatility - GARCH(1,1)')
plt.xlabel('Fecha')
plt.ylabel('Volatility')
plt.show()

In [ ]:
sigma_garch = df['Volatility'][-1]
sigma_garch

In [ ]:
r_avg = np.mean(df['Log Returns'])*252
r_avg

## Geometric Brownian Motion (GBM) - 1 path

### 10.1. Parámetros  

In [ ]:
# Define the parameters of the stochastic process
T = 8/12       # Time horizon
N = 168     # Number of time steps
dt = T / N    # Time step size

sigma = sigma_garch    # Volatility of the process
mu = r_avg + (1/2)*sigma_garch**2   # Mean of the process
div = 0 # Dividend yield
r = 0.0363 # r: risk-free interest rate
K = 250 # K: strike price

# Define the initial condition
S0 = df['Close'][-1]

### 10.2. Geometric Brownian Motion (GBM) - 1 path

In [ ]:
# Define the Wiener process
dw = np.random.normal(0, np.sqrt(dt), N)

# Define the time grid
t = np.linspace(0, T, N+1)

# Define the log-normal process
S = np.zeros(N+1)
S[0] = S0
for i in range(1, N+1):
    S[i] = S[i-1] * np.exp((mu - 0.5*sigma**2) * dt + sigma * dw[i-1])

# Define the derivative of the process
dS = np.zeros(N+1)
for i in range(1, N+1):
    dS[i] = (mu+0.5*sigma**2)*dt + sigma*dw[i-1]

In [ ]:
# Plot the results
plt.figure(figsize=(10, 6))
plt.subplot(2, 1, 1)
plt.plot(t, S)
plt.ylabel('S')
plt.ylim(0, max(S)*1.1)
plt.grid(True)
plt.subplot(2, 1, 2)
plt.plot(t, dS)
plt.ylabel('dS')
plt.grid(True)
plt.show()

### 10.3. Geometric Brownian Motion (GBM) - R paths

In [ ]:
def gbm_paths(S0, mu, sigma, T, N, num_paths):
    dt = T/N
    paths = np.zeros((num_paths, N+1))
    paths[:, 0] = S0

    for i in range(num_paths):
        for j in range(1, N+1):
            drift = (mu - 0.5 * sigma**2) * dt
            diffusion = sigma * np.sqrt(dt) * np.random.normal()
            paths[i, j] = paths[i, j-1] * np.exp(drift + diffusion)

    return paths

In [ ]:
num_paths = 20000

paths = gbm_paths(S0, mu, sigma, T, N, num_paths)

plt.figure(figsize=(10, 6))
plt.plot(paths.T)
plt.xlabel('Time Steps')
plt.ylabel('Stock Price')
plt.title('Geometric Brownian Motion')
plt.show()

### 10.4 Call option price

In [ ]:
# calculate the call option payoff at maturity
payoff = np.maximum(paths[:, -1] - K, 0)
# calculate the call option price
call_price = np.exp(-r * T) * np.mean(payoff)
call_price

### 10.5 Put option price

In [ ]:
# calculate the put option payoff at maturity
payoff = np.maximum(K - paths[:, -1], 0)
# calculate the put option price
put_price = np.exp(-r * T) * np.mean(payoff)
put_price

## 11. Geometric Brownian Motion (GBM) - Risk neutral

Geometric Brownian Motion (GBM) - RN

In [ ]:
def gbm_paths_RN(S0, mu, sigma, T, N, num_paths):
    dt = T/N
    paths = np.zeros((num_paths, N+1))
    paths[:, 0] = S0

    for i in range(num_paths):
        for j in range(1, N+1):
            drift = ((r-div) - 0.5 * sigma**2) * dt
            diffusion = sigma * np.sqrt(dt) * np.random.normal()
            paths[i, j] = paths[i, j-1] * np.exp(drift + diffusion)

    return paths

### 10.2 Paths - RN

In [ ]:
num_paths = 100000

paths_RN = gbm_paths_RN(S0, mu, sigma, T, N, num_paths)

plt.figure(figsize=(10, 6))
plt.plot(paths.T)
plt.xlabel('Time Steps')
plt.ylabel('Stock Price')
plt.title('Geometric Brownian Motion')
plt.show()

### 11.3 Call option price - RN

In [ ]:
# calculate the call option payoff at maturity
payoff = np.maximum(paths_RN[:, -1] - K, 0)
# calculate the call option price
call_price = np.exp(-r * T) * np.mean(payoff)
call_price

### 11.4 Put option price - RN

In [ ]:
# calculate the put option payoff at maturity
payoff = np.maximum(K - paths_RN[:, -1], 0)
# calculate the put option price
put_price = np.exp(-r * T) * np.mean(payoff)
put_price

In [ ]:
# Black-Scholes-Merton formula for a call option
def black_scholes_merton_call(S, K, T, r, sigma, div):
    d1 = (np.log(S / K) + (r - div + 0.5 * sigma**2) * T) / (sigma * np.sqrt(T))
    d2 = d1 - sigma * np.sqrt(T)
    call_price = S * np.exp(-div * T) * norm.cdf(d1) - K * np.exp(-r * T) * norm.cdf(d2)
    return call_price

# Calculate the call option price using BSM
bsm_call_price = black_scholes_merton_call(S0, K, T, r, sigma, div)
print("Black-Scholes-Merton Call Option Price:", bsm_call_price)

In [ ]:
# Black-Scholes-Merton formula for a put option
def black_scholes_merton_put(S, K, T, r, sigma, div):
    d1 = (np.log(S / K) + (r - div + 0.5 * sigma**2) * T) / (sigma * np.sqrt(T))
    d2 = d1 - sigma * np.sqrt(T)
    put_price = K * np.exp(-r * T) * norm.cdf(-d2) - S * np.exp(-div * T) * norm.cdf(-d1)
    return put_price

# Calculate the put option price using BSM
bsm_put_price = black_scholes_merton_put(S0, K, T, r, sigma, div)
print("Black-Scholes-Merton Put Option Price:", bsm_put_price)